# **Kaggle – DataTops®**
Tu TA ha decidido cambiar de aires y, por eso, ha comprado una tienda de portátiles. Sin embargo, su única especialidad es Data Science, por lo que ha decidido crear un modelo de ML para establecer los mejores precios.

¿Podrías ayudar a tu profe a mejorar ese modelo?

## Aspectos importantes
- Última submission:
    - Mañana: 17 de febrero a las 5pm
    - Tarde: 19 de febrero a las 5pm
- **Enlace de la competición**: https://www.kaggle.com/t/c5cc87b50c4b4770bdc8f5acbe15577d
- **Requisito**: Estar registrado en [Kaggle](https://www.kaggle.com/)

## Métrica:
El error cuadrático medio (RMSE, por sus siglas en inglés) es una medida de la desviación estándar de los residuos (errores de predicción). Los residuos representan la diferencia entre los valores observados y los valores predichos por el modelo. El RMSE indica qué tan dispersos están estos errores: cuanto menor es el RMSE, más cercanas están las predicciones a los valores reales. En otras palabras, el RMSE mide qué tan bien se ajusta la línea de regresión a los datos.


$$ RMSE = \sqrt{\frac{1}{n}\Sigma_{i=1}^{n}{\Big(\frac{d_i -f_i}{\sigma_i}\Big)^2}}$$


## 1. Librerías

In [1]:
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
import urllib.request

## 2. Datos

In [2]:
# Para que funcione necesitas bajarte los archivos de datos de Kaggle
df = pd.read_csv("./data/train.csv",index_col= "laptop_ID")

### 2.1 Exploración de los datos

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 912 entries, 755 to 229
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Company           912 non-null    object 
 1   Product           912 non-null    object 
 2   TypeName          912 non-null    object 
 3   Inches            912 non-null    float64
 4   ScreenResolution  912 non-null    object 
 5   Cpu               912 non-null    object 
 6   Ram               912 non-null    object 
 7   Memory            912 non-null    object 
 8   Gpu               912 non-null    object 
 9   OpSys             912 non-null    object 
 10  Weight            912 non-null    object 
 11  Price_in_euros    912 non-null    float64
dtypes: float64(2), object(10)
memory usage: 92.6+ KB


In [4]:
df.head()

,Company,Product,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight,Price_in_euros
laptop_ID,,,,,,,,,,,,
755,HP,250 G6,Notebook,15.6,Full HD 1920x1080,Intel Core i3 6006U 2GHz,8GB,256GB SSD,Intel HD Graphics 520,Windows 10,1.86kg,539.00
618,Dell,Inspiron 7559,Gaming,15.6,Full HD 1920x1080,Intel Core i7 6700HQ 2.6GHz,16GB,1TB HDD,Nvidia GeForce GTX 960<U+039C>,Windows 10,2.59kg,879.01
909,HP,ProBook 450,Notebook,15.6,Full HD 1920x1080,Intel Core i7 7500U 2.7GHz,8GB,1TB HDD,Nvidia GeForce 930MX,Windows 10,2.04kg,900.00
2,Apple,Macbook Air,Ultrabook,13.3,1440x900,Intel Core i5 1.8GHz,8GB,128GB Flash Storage,Intel HD Graphics 6000,macOS,1.34kg,898.94
286,Dell,Inspiron 3567,Notebook,15.6,Full HD 1920x1080,Intel Core i3 6006U 2.0GHz,4GB,1TB HDD,AMD Radeon R5 M430,Linux,2.25kg,428.00


In [5]:
'''
Antes de continuar debemos hacer una transformacion de los datos para que el modelo permita su correcto funcionamiento. Ahora transformaremos con la siguiente logica:
One-hot encoding(get_dummies) = Cuando la variable no guarde una jerarquia como tal 
Ordinal encoding = Cuando la variable si que guarde una jerarquia
Aplicaremos todo esto a todos los objetos que en la descripcion aparezcan como objects

'''

'\nAntes de continuar debemos hacer una transformacion de los datos para que el modelo permita su correcto funcionamiento. Ahora transformaremos con la siguiente logica:\nOne-hot encoding(get_dummies) = Cuando la variable no guarde una jerarquia como tal \nOrdinal encoding = Cuando la variable si que guarde una jerarquia\nAplicaremos todo esto a todos los objetos que en la descripcion aparezcan como objects\n\n'

In [6]:
ordinal_cols = ["ScreenResolution","Cpu","Ram","Memory","Weight"]
one_hot_cols = ["Company", "Product","TypeName","Gpu","OpSys", "ScreenResolution","Cpu","Ram","Memory","Weight"]


In [7]:
import category_encoders as ce
encoder = ce.TargetEncoder(cols= one_hot_cols)
df[one_hot_cols] = encoder.fit_transform(df[["Company", "Product","TypeName","Gpu","OpSys", "ScreenResolution","Cpu","Ram","Memory","Weight"]], df["Price_in_euros"])

In [8]:
df

,Company,Product,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight,Price_in_euros
laptop_ID,,,,,,,,,,,,
755,1093.484640,896.754608,766.723674,15.6,1168.386533,553.022980,1181.682765,1224.399362,1070.603692,1161.645951,838.783656,539.00
618,1173.998679,1081.446015,1701.049555,15.6,1168.386533,1510.534268,1938.087425,681.252310,1106.586373,1161.645951,1177.283687,879.01
909,1093.484640,1026.054431,766.723674,15.6,1168.386533,1332.186766,1181.682765,681.252310,1091.328884,1161.645951,983.545937,900.00
2,1294.117288,1099.964056,1552.960384,13.3,1099.450666,1099.964056,1181.682765,1043.903346,1095.488932,1277.758506,1109.403577,898.94
286,1173.998679,812.965790,766.723674,15.6,1168.386533,929.473941,573.200824,681.252310,876.735845,633.995890,1004.834182,428.00
...,...,...,...,...,...,...,...,...,...,...,...,...
28,1173.998679,1049.932354,766.723674,15.6,1168.386533,977.901814,1181.682765,1224.399362,974.064902,1161.645951,693.969087,800.00
1160,1093.484640,1215.979274,1337.084762,13.3,1317.848915,1175.147063,1181.682765,1224.399362,1070.603692,1161.645951,1208.833651,1629.00
78,1053.578417,1012.451308,766.723674,15.6,1168.386533,902.337743,1181.682765,985.760407,1142.610322,641.531917,693.969087,519.00


In [9]:
df.tail()

,Company,Product,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight,Price_in_euros
laptop_ID,,,,,,,,,,,,
28,1173.998679,1049.932354,766.723674,15.6,1168.386533,977.901814,1181.682765,1224.399362,974.064902,1161.645951,693.969087,800.00
1160,1093.484640,1215.979274,1337.084762,13.3,1317.848915,1175.147063,1181.682765,1224.399362,1070.603692,1161.645951,1208.833651,1629.00
78,1053.578417,1012.451308,766.723674,15.6,1168.386533,902.337743,1181.682765,985.760407,1142.610322,641.531917,693.969087,519.00
23,1093.484640,994.346848,766.723674,15.6,524.277870,997.076642,573.200824,627.902845,960.506814,641.531917,838.783656,258.00
229,1173.998679,1594.351818,1701.049555,17.3,1368.960394,1742.461379,1938.087425,1766.188917,1653.728041,1161.645951,1516.935556,2456.34


In [10]:
df.describe()

,Company,Product,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight,Price_in_euros
count,912.000000,912.000000,912.000000,912.000000,912.000000,912.000000,912.000000,912.000000,912.000000,912.000000,912.000000,912.000000
mean,1093.152727,1111.780525,1105.932235,14.981579,1062.300220,1135.497977,1102.694602,1101.912200,1105.819671,1110.504883,1046.776350,1111.724090
std,187.338983,131.915701,405.197281,1.436719,310.502754,316.092546,447.230899,402.124469,242.834080,183.261038,188.440593,687.959172
min,609.880132,812.965790,766.723674,10.100000,524.277870,553.022980,573.200824,479.728331,520.560328,633.995890,658.488955,174.000000
25%,1053.578417,1029.401324,766.723674,14.000000,943.961656,902.337743,573.200824,681.252310,1022.754498,1161.645951,970.197702,589.000000
50%,1093.484640,1085.607301,766.723674,15.600000,1168.386533,1085.440319,1181.682765,1224.399362,1123.962420,1161.645951,1080.719985,978.000000
75%,1173.998679,1179.226260,1552.960384,15.600000,1283.295236,1332.186766,1181.682765,1224.399362,1225.131403,1161.645951,1164.636845,1483.942500
max,1620.483224,1594.351818,1701.049555,18.400000,1499.618281,1742.461379,1938.087425,1819.083001,1898.935670,1518.059712,1760.610950,6099.000000


### 2.3 Definir X e y

In [11]:
X = df.drop(['Price_in_euros'], axis=1)
y = df['Price_in_euros'].copy()
X_train = df.drop(['Price_in_euros'], axis=1)
y_train = df['Price_in_euros'].copy()
X.shape

(912, 11)

In [12]:
y.shape

(912,)

### 2.4 Dividir X_train, X_test, y_train, y_test

In [13]:
X_train

,Company,Product,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight
laptop_ID,,,,,,,,,,,
755,1093.484640,896.754608,766.723674,15.6,1168.386533,553.022980,1181.682765,1224.399362,1070.603692,1161.645951,838.783656
618,1173.998679,1081.446015,1701.049555,15.6,1168.386533,1510.534268,1938.087425,681.252310,1106.586373,1161.645951,1177.283687
909,1093.484640,1026.054431,766.723674,15.6,1168.386533,1332.186766,1181.682765,681.252310,1091.328884,1161.645951,983.545937
2,1294.117288,1099.964056,1552.960384,13.3,1099.450666,1099.964056,1181.682765,1043.903346,1095.488932,1277.758506,1109.403577
286,1173.998679,812.965790,766.723674,15.6,1168.386533,929.473941,573.200824,681.252310,876.735845,633.995890,1004.834182
...,...,...,...,...,...,...,...,...,...,...,...
28,1173.998679,1049.932354,766.723674,15.6,1168.386533,977.901814,1181.682765,1224.399362,974.064902,1161.645951,693.969087
1160,1093.484640,1215.979274,1337.084762,13.3,1317.848915,1175.147063,1181.682765,1224.399362,1070.603692,1161.645951,1208.833651
78,1053.578417,1012.451308,766.723674,15.6,1168.386533,902.337743,1181.682765,985.760407,1142.610322,641.531917,693.969087


In [14]:
y_train

laptop_ID
755      539.00
618      879.01
909      900.00
2        898.94
286      428.00
         ...   
28       800.00
1160    1629.00
78       519.00
23       258.00
229     2456.34
Name: Price_in_euros, Length: 912, dtype: float64

## 3. Procesado de datos

Nuestro target es la columna `Price_in_euros`

In [15]:
target = "Price_in_euros"

In [16]:
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor


In [17]:
from sklearn.model_selection import cross_val_score

-----------------------------------------------------------------------------------------------------------------

## 4. Modelado

### 4.1 Baseline de modelos


In [18]:
rf_reg = RandomForestRegressor(max_depth = 10, random_state= 42)
lgb_reg = LGBMRegressor(max_depth = 10, random_state = 42, verbose = -1)
xgb_reg = XGBRegressor(max_depth = 10, random_state = 42)

modelos_reg = {
    "Random Forest": rf_reg,
    "LightGBM": lgb_reg,
    "XGBoost Regressor": xgb_reg
}

### 4.2 Sacar métricas, valorar los modelos

Recuerda que en la competición se va a evaluar con la métrica de ``RMSE``.

In [19]:
for nombre, modelo in zip(["XGBRegressor","LGBMRegressor","RandomForestRegressor"],[xgb_reg, lgb_reg, rf_reg]):
    print(f"Para {nombre}:", end = " ")
    print("neg_mean_absolute error:", np.mean(cross_val_score(modelo, X_train, y_train, cv = 5, scoring = "neg_root_mean_squared_error")))

Para XGBRegressor: neg_mean_absolute error: -206.14028292958474
Para LGBMRegressor: neg_mean_absolute error: -193.9655320861186
Para RandomForestRegressor: neg_mean_absolute error: -189.71355839847115


In [20]:
def objective(trial):
    param_grid = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 800),
        "max_depth": trial.suggest_int("max_depth", 3, 70),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
        "criterion": trial.suggest_categorical("criterion", ["squared_error", "absolute_error"]),
        "max_samples": trial.suggest_float("max_samples", 0.5, 1.0)
    }

    model = XGBRegressor(**param_grid, random_state=42, n_jobs=-1)

    score = cross_val_score(
        model,
        X_train,
        y_train,
        cv=5,
        scoring="neg_root_mean_squared_error",
        n_jobs=-1
    ).mean()

    return score

In [21]:
import optuna
study = optuna.create_study(direction="maximize", 
                            sampler=optuna.samplers.TPESampler(seed=42))

c:\Users\Dani\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-02-18 18:06:42,698] A new study created in memory with name: no-name-5c0ea6d2-ee94-450e-98fc-5be45254bd96


In [22]:
study.optimize(objective, n_trials=50)

[I 2026-02-18 18:06:45,876] Trial 0 finished with value: -204.31431764472254 and parameters: {'n_estimators': 362, 'max_depth': 67, 'min_samples_split': 15, 'min_samples_leaf': 6, 'max_features': 'sqrt', 'criterion': 'squared_error', 'max_samples': 0.8540362888980227}. Best is trial 0 with value: -204.31431764472254.
[I 2026-02-18 18:06:48,602] Trial 1 finished with value: -204.31431764472254 and parameters: {'n_estimators': 114, 'max_depth': 68, 'min_samples_split': 17, 'min_samples_leaf': 3, 'max_features': None, 'criterion': 'squared_error', 'max_samples': 0.645614570099021}. Best is trial 0 with value: -204.31431764472254.
[I 2026-02-18 18:06:50,921] Trial 2 finished with value: -204.57960279831997 and parameters: {'n_estimators': 528, 'max_depth': 12, 'min_samples_split': 7, 'min_samples_leaf': 4, 'max_features': 'log2', 'criterion': 'absolute_error', 'max_samples': 0.5232252063599989}. Best is trial 0 with value: -204.31431764472254.
[I 2026-02-18 18:06:53,087] Trial 3 finished w

In [23]:
optuna.visualization.plot_param_importances(study)

In [24]:
optuna.visualization.plot_optimization_history(study)

In [25]:
best_params = study.best_params

In [26]:
best_n_estimators = best_params["n_estimators"]
best_max_depth = best_params["max_depth"]
best_min_samples_split = best_params["min_samples_split"]
best_min_samples_leaf = best_params["min_samples_leaf"]
best_max_features = best_params["max_features"]
best_criterion = best_params["criterion"]
best_max_samples = best_params["max_samples"]

In [27]:
best_model = XGBRegressor(
    n_estimators=best_n_estimators,
    max_depth=best_max_depth,
    min_samples_split=best_min_samples_split,
    min_samples_leaf=best_min_samples_leaf,
    max_features=best_max_features,
    criterion=best_criterion,
    max_samples=best_max_samples,
    random_state=42,
    n_jobs=-1
)

In [28]:
best_model.fit(X_train,y_train)

c:\Users\Dani\AppData\Local\Programs\Python\Python311\Lib\site-packages\xgboost\training.py:199: UserWarning:

[18:07:11] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "criterion", "max_features", "max_samples", "min_samples_leaf", "min_samples_split" } are not used.




,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes

### 4.3 Optimización (up to you 🫰🏻)

-----------------------------------------------------------------

## Una vez listo el modelo, toca predecir ``test.csv``

**RECUERDA: APLICAR LAS TRANSFORMACIONES QUE HAYAS REALIZADO EN `train.csv` a `test.csv`.**


Véase:
- Estandarización/Normalización
- Eliminación de Outliers
- Eliminación de columnas
- Creación de columnas nuevas
- Gestión de valores nulos
- Y un largo etcétera de técnicas que como Data Scientist hayas considerado las mejores para tu dataset.

## 1. Carga los datos de `test.csv` para predecir.


In [29]:
X_pred = pd.read_csv("./data/test.csv", index_col = "laptop_ID")
X_pred.head()

,Company,Product,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight
laptop_ID,,,,,,,,,,,
209,Lenovo,Legion Y520-15IKBN,Gaming,15.6,Full HD 1920x1080,Intel Core i7 7700HQ 2.8GHz,16GB,512GB SSD,Nvidia GeForce GTX 1060,No OS,2.4kg
1281,Acer,Aspire ES1-531,Notebook,15.6,1366x768,Intel Celeron Dual Core N3060 1.6GHz,4GB,500GB HDD,Intel HD Graphics 400,Linux,2.4kg
1168,Lenovo,V110-15ISK (i3-6006U/4GB/1TB/No,Notebook,15.6,1366x768,Intel Core i3 6006U 2.0GHz,4GB,1TB HDD,Intel HD Graphics 520,No OS,1.9kg
1231,Dell,Inspiron 7579,2 in 1 Convertible,15.6,IPS Panel Full HD / Touchscreen 1920x1080,Intel Core i5 7200U 2.5GHz,8GB,256GB SSD,Intel HD Graphics 620,Windows 10,2.191kg
1020,HP,ProBook 640,Notebook,14.0,Full HD 1920x1080,Intel Core i5 7200U 2.5GHz,4GB,256GB SSD,Intel HD Graphics 620,Windows 10,1.95kg


In [30]:
X_pred.tail()

,Company,Product,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight
laptop_ID,,,,,,,,,,,
820,MSI,GE72MVR 7RG,Gaming,17.3,Full HD 1920x1080,Intel Core i7 7700HQ 2.8GHz,16GB,512GB SSD + 1TB HDD,Nvidia GeForce GTX 1070,Windows 10,2.9kg
948,Toshiba,Tecra Z40-C-12X,Notebook,14.0,IPS Panel Full HD 1920x1080,Intel Core i5 6200U 2.3GHz,4GB,128GB SSD,Intel HD Graphics 520,Windows 10,1.47kg
483,Dell,Precision M5520,Workstation,15.6,Full HD 1920x1080,Intel Core i7 7700HQ 2.8GHz,8GB,256GB SSD,Nvidia Quadro M1200,Windows 10,1.78kg
1017,HP,Probook 440,Notebook,14.0,1366x768,Intel Core i5 7200U 2.5GHz,4GB,500GB HDD,Intel HD Graphics 620,Windows 10,1.64kg
421,Asus,ZenBook Flip,2 in 1 Convertible,13.3,IPS Panel Full HD / Touchscreen 1920x1080,Intel Core i5 7200U 2.5GHz,8GB,256GB SSD,Intel HD Graphics 620,Windows 10,1.27kg


In [31]:
X_pred.info()

<class 'pandas.core.frame.DataFrame'>
Index: 391 entries, 209 to 421
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Company           391 non-null    object 
 1   Product           391 non-null    object 
 2   TypeName          391 non-null    object 
 3   Inches            391 non-null    float64
 4   ScreenResolution  391 non-null    object 
 5   Cpu               391 non-null    object 
 6   Ram               391 non-null    object 
 7   Memory            391 non-null    object 
 8   Gpu               391 non-null    object 
 9   OpSys             391 non-null    object 
 10  Weight            391 non-null    object 
dtypes: float64(1), object(10)
memory usage: 36.7+ KB


 ## 2. Replicar el procesado para ``test.csv``

In [32]:
X_pred

,Company,Product,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight
laptop_ID,,,,,,,,,,,
209,Lenovo,Legion Y520-15IKBN,Gaming,15.6,Full HD 1920x1080,Intel Core i7 7700HQ 2.8GHz,16GB,512GB SSD,Nvidia GeForce GTX 1060,No OS,2.4kg
1281,Acer,Aspire ES1-531,Notebook,15.6,1366x768,Intel Celeron Dual Core N3060 1.6GHz,4GB,500GB HDD,Intel HD Graphics 400,Linux,2.4kg
1168,Lenovo,V110-15ISK (i3-6006U/4GB/1TB/No,Notebook,15.6,1366x768,Intel Core i3 6006U 2.0GHz,4GB,1TB HDD,Intel HD Graphics 520,No OS,1.9kg
1231,Dell,Inspiron 7579,2 in 1 Convertible,15.6,IPS Panel Full HD / Touchscreen 1920x1080,Intel Core i5 7200U 2.5GHz,8GB,256GB SSD,Intel HD Graphics 620,Windows 10,2.191kg
1020,HP,ProBook 640,Notebook,14.0,Full HD 1920x1080,Intel Core i5 7200U 2.5GHz,4GB,256GB SSD,Intel HD Graphics 620,Windows 10,1.95kg
...,...,...,...,...,...,...,...,...,...,...,...
820,MSI,GE72MVR 7RG,Gaming,17.3,Full HD 1920x1080,Intel Core i7 7700HQ 2.8GHz,16GB,512GB SSD + 1TB HDD,Nvidia GeForce GTX 1070,Windows 10,2.9kg
948,Toshiba,Tecra Z40-C-12X,Notebook,14.0,IPS Panel Full HD 1920x1080,Intel Core i5 6200U 2.3GHz,4GB,128GB SSD,Intel HD Graphics 520,Windows 10,1.47kg
483,Dell,Precision M5520,Workstation,15.6,Full HD 1920x1080,Intel Core i7 7700HQ 2.8GHz,8GB,256GB SSD,Nvidia Quadro M1200,Windows 10,1.78kg


In [33]:
X_pred[one_hot_cols] = encoder.transform(X_pred[one_hot_cols])

In [34]:
X_pred

,Company,Product,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight
laptop_ID,,,,,,,,,,,
209,1053.578417,1076.414942,1701.049555,15.6,1168.386533,1742.461379,1938.087425,1819.083001,1653.728041,641.531917,884.587761
1281,609.880132,995.019802,766.723674,15.6,524.277870,604.011346,573.200824,627.902845,520.560328,633.995890,884.587761
1168,1053.578417,1111.724090,766.723674,15.6,524.277870,929.473941,573.200824,681.252310,1070.603692,641.531917,1042.851395
1231,1173.998679,1111.724090,1337.084762,15.6,1175.119678,902.337743,1181.682765,1224.399362,1142.610322,1161.645951,1111.724090
1020,1093.484640,1104.197910,766.723674,14.0,1168.386533,902.337743,573.200824,1224.399362,1142.610322,1161.645951,1282.943032
...,...,...,...,...,...,...,...,...,...,...,...
820,1620.483224,1231.069459,1701.049555,17.3,1168.386533,1742.461379,1938.087425,1456.465259,1898.935670,1161.645951,1268.883895
948,1243.951857,1111.724090,766.723674,14.0,1368.960394,1085.440319,573.200824,714.437637,1070.603692,1161.645951,1113.110813
483,1173.998679,1111.724090,1672.825045,15.6,1168.386533,1742.461379,1181.682765,1224.399362,1295.335363,1161.645951,1111.724090


In [35]:
predictions_submit = best_model.predict(X_pred)
predictions_submit

array([1369.1355 ,  273.55753,  755.7109 ,  981.28094,  988.3595 ,
        827.0373 , 1070.7388 ,  877.0437 , 1033.911  ,  979.09314,
       1219.7153 , 1924.1322 ,  905.5639 , 1342.789  ,  973.56726,
        988.1421 , 1434.1158 , 1297.2649 , 1801.5103 ,  643.9036 ,
       1505.1993 , 1016.6183 ,  877.09174, 1108.8268 ,  457.44217,
        708.5711 ,  936.04517, 1109.4972 , 3028.1262 , 1059.0636 ,
       1305.5856 ,  418.21683,  850.30383, 3296.9775 , 1311.6945 ,
       1139.4491 ,  647.82056, 1213.0134 ,  865.8817 , 1691.176  ,
        859.8629 ,  734.81433,  530.1698 , 1139.6819 , 1199.1866 ,
       1213.6136 , 1167.2675 ,  629.6721 ,  562.0112 ,  358.41962,
       1352.0566 ,  673.94385, 1107.8113 ,  451.78903, 1892.8158 ,
       1947.2926 , 1025.497  , 1063.6649 ,  709.1747 ,  939.60004,
       1839.4458 , 2017.2295 ,  815.5244 , 2135.902  , 1804.5444 ,
       1132.6383 , 1163.8589 , 1005.1263 , 2258.9211 , 1660.1747 ,
        898.3565 , 1036.7646 ,  972.7255 , 1855.935  , 1141.22

**¡OJO! ¿Por qué me da error?**

IMPORTANTE:

- SI EL ARRAY CON EL QUE HICISTEIS `.fit()` ERA DE 4 COLUMNAS, PARA `.predict()` DEBEN SER LAS MISMAS
- SI AL ARRAY CON EL QUE HICISTEIS `.fit()` LO NORMALIZASTEIS, PARA `.predict()` DEBÉIS NORMALIZARLO
- TODO IGUAL SALVO **BORRAR FILAS**, EL NÚMERO DE ROWS SE DEBE MANTENER EN ESTE SET, PUES LA PREDICCIÓN DEBE TENER **391 FILAS**, SI O SI

**Entonces, si al cargar los datos de ``train.csv`` usaste `index_col=0`, ¿tendré que hacer lo también para el `test.csv`?**

In [36]:
# ¿Qué opináis?
# ¿Sí, no?

![wow.jpeg](attachment:wow.jpeg)

## 3. **¿Qué es lo que subirás a Kaggle?**

**Para subir a Kaggle la predicción esta tendrá que tener una forma específica.**

En este caso, la **MISMA** forma que `sample_submission.csv`.

In [37]:
sample = pd.read_csv("data/sample_submission.csv")

In [38]:
sample.head()

,laptop_ID,Price_in_euros
0,209,1949.1
1,1281,805.0
2,1168,1101.0
3,1231,1293.8
4,1020,1832.6


In [39]:
sample.shape

(391, 2)

## 4. Mete tus predicciones en un dataframe llamado ``submission``.

In [40]:
#¿Cómo creamos la submission?
submission = pd.DataFrame({"laptop_ID": X_pred.index , "Price_in_euros" : predictions_submit})

In [41]:
submission.head()

,laptop_ID,Price_in_euros
0,209,1369.135498
1,1281,273.557526
2,1168,755.710876
3,1231,981.280945
4,1020,988.359497


In [42]:
submission.shape

(391, 2)

## 5. Pásale el CHEQUEADOR para comprobar que efectivamente está listo para subir a Kaggle.

In [43]:
def chequeador(df_to_submit):
    """
    Esta función se asegura de que tu submission tenga la forma requerida por Kaggle.

    Si es así, se guardará el dataframe en un `csv` y estará listo para subir a Kaggle.

    Si no, LEE EL MENSAJE Y HAZLE CASO.

    Si aún no:
    - apaga tu ordenador,
    - date una vuelta,
    - enciendelo otra vez,
    - abre este notebook y
    - leelo todo de nuevo.
    Todos nos merecemos una segunda oportunidad. También tú.
    """
    if df_to_submit.shape == sample.shape:
        if df_to_submit.columns.all() == sample.columns.all():
            if df_to_submit.laptop_ID.all() == sample.laptop_ID.all():
                print("You're ready to submit!")
                df_to_submit.to_csv("submission.csv", index = False) #muy importante el index = False
                urllib.request.urlretrieve("https://www.mihaileric.com/static/evaluation-meme-e0a350f278a36346e6d46b139b1d0da0-ed51e.jpg", "gfg.png")
                img = Image.open("gfg.png")
                img.show()
            else:
                print("Check the ids and try again")
        else:
            print("Check the names of the columns and try again")
    else:
        print("Check the number of rows and/or columns and try again")
        print("\nMensaje secreto del TA: No me puedo creer que después de todo este notebook hayas hecho algún cambio en las filas de `test.csv`. Lloro.")

In [44]:
chequeador(submission)

You're ready to submit!


In [50]:
y_test = X_pred[target]

KeyError: 'Price_in_euros'

In [51]:
from sklearn.metrics import mean_squared_error
y_pred = best_model.predict(X_pred)
print(round(mean_squared_error(y_test, y_pred),4))

NameError: name 'y_test' is not defined